## Full Images

In [21]:
# ============================================================
# RESPONSE-ONLY MAGNETIC BEAD COUNTER
#
# Core rule:
#   1. Build the same local-dark response used by the earlier code.
#   2. Detect beads ONLY from strong, compact spots in that response.
#   3. Use the original image only for the final overlay.
#   4. FOV/artifact masks restrict valid pixels.
#   5. Focus/clutter masks only select sensitivity. They are NOT bead masks.
#
# INPUT IMAGE FOLDERS
#     ./images_n45/
#     ./images_n52/
#
# MASK FOLDERS
#     ./annotate_n4552/labeled_images/fov/
#     ./annotate_n4552/labeled_images/artifact/
#     ./annotate_n4552/labeled_images/clutter/
#     ./annotate_n4552/labeled_images/focus/
#
# OUTPUT
#     ./results_n4552_response_strong/
# ============================================================

from pathlib import Path
import re
import time
import traceback

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from skimage.feature import blob_log


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(".")

DATASETS = {
    "n45": PROJECT_DIR / "images_n45",
    "n52": PROJECT_DIR / "images_n52",
}

LABEL_DIR = PROJECT_DIR / "annotate_n4552" / "labeled_images"
FOV_MASK_DIR = LABEL_DIR / "fov"
ARTIFACT_MASK_DIR = LABEL_DIR / "artifact"
CLUTTER_MASK_DIR = LABEL_DIR / "clutter"
FOCUS_MASK_DIR = LABEL_DIR / "focus"

RESULTS_DIR = PROJECT_DIR / "results_n4552_response_strong"
OVERLAY_DIR = RESULTS_DIR / "overlays"
BEAD_MASK_DIR = RESULTS_DIR / "bead_masks"
DETECTION_TABLE_DIR = RESULTS_DIR / "detection_tables"
DEBUG_DIR = RESULTS_DIR / "debug"
FINAL_EXCEL_PATH = RESULTS_DIR / "final_counts.xlsx"


# ============================================================
# DEBUG OR BATCH MODE
# ============================================================

DEBUG_MODE = False
DEBUG_IMAGE_PATH = Path("./images_n52/n52 - S9-2nd Wash.jpg")
SHOW_DEBUG_IMAGES = True
SAVE_DEBUG_IMAGES = True


# ============================================================
# MASK SETTINGS
# ============================================================

USE_FOV_MASK = True
USE_ARTIFACT_MASK = True
USE_CLUTTER_MASK = True
USE_FOCUS_MASK = True

SKIP_IF_FOV_MISSING = True
CONTINUE_IF_ARTIFACT_MISSING = True
CONTINUE_IF_CLUTTER_MISSING = True
CONTINUE_IF_FOCUS_MISSING = True

FOV_MASK_SUFFIX = "_fov_mask"
ARTIFACT_MASK_SUFFIX = "_artifact_mask"
CLUTTER_MASK_SUFFIX = "_clutter_mask"
FOCUS_MASK_SUFFIX = "_focus_mask"

FOV_INNER_MARGIN_PX = 1


# ============================================================
# PHYSICAL SCALE
# ============================================================

SCALE_PIXELS = 135.0
SCALE_MICRONS = 500.0
MICRONS_PER_PIXEL = SCALE_MICRONS / SCALE_PIXELS
PIXELS_PER_MICRON = SCALE_PIXELS / SCALE_MICRONS

MIN_BEAD_DIAMETER_UM = 10.0
MAX_BEAD_DIAMETER_UM = 35.0
MIN_BEAD_DIAMETER_PX = MIN_BEAD_DIAMETER_UM * PIXELS_PER_MICRON
MAX_BEAD_DIAMETER_PX = MAX_BEAD_DIAMETER_UM * PIXELS_PER_MICRON
MIN_BEAD_RADIUS_PX = MIN_BEAD_DIAMETER_PX / 2.0
MAX_BEAD_RADIUS_PX = MAX_BEAD_DIAMETER_PX / 2.0

print(f"Scale: {MICRONS_PER_PIXEL:.4f} µm/pixel")
print(
    "Expected bead diameter: "
    f"{MIN_BEAD_DIAMETER_PX:.2f}–{MAX_BEAD_DIAMETER_PX:.2f} pixels"
)


# ============================================================
# RESPONSE CREATION
# ============================================================

GAUSSIAN_SIGMA = 0.55
BACKGROUND_KERNEL = 31

# Display scaling only. Detection uses the raw float response.
RESPONSE_DISPLAY_HIGH_PERCENTILE = 99.80


# ============================================================
# RESPONSE-ONLY DETECTOR SETTINGS
# ============================================================

# LoG size range, derived from expected bead size.
DETECT_MIN_SIGMA = max(
    0.50,
    MIN_BEAD_DIAMETER_PX / (2.0 * np.sqrt(2.0)),
)
DETECT_MAX_SIGMA = max(
    DETECT_MIN_SIGMA + 0.20,
    MAX_BEAD_DIAMETER_PX / (2.0 * np.sqrt(2.0)),
)
DETECT_NUM_SIGMA = 10
DETECT_OVERLAP = 0.80

# Thresholds are expressed on response / 255, exactly like the successful
# normal LoG detector in the original code. These values prevent faint
# response speckle from becoming candidates.
REGION_SETTINGS = {
    "normal": {
        "log_threshold": 0.026,
        "min_raw_peak": 5.5,
        "min_center_mean": 3.5,
        "min_center_minus_ring": 1.15,
        "min_center_ring_ratio": 1.35,
        "min_display_value": 42,
        "min_hessian_ratio": 0.16,
        "max_support_axis_ratio": 3.2,
        "max_opposite_line_pairs": 1,
    },
    "clutter": {
        "log_threshold": 0.020,
        "min_raw_peak": 4.5,
        "min_center_mean": 2.8,
        "min_center_minus_ring": 0.80,
        "min_center_ring_ratio": 1.25,
        "min_display_value": 34,
        "min_hessian_ratio": 0.12,
        "max_support_axis_ratio": 3.8,
        "max_opposite_line_pairs": 1,
    },
    "focus": {
        # Focus is more sensitive, but still requires a visibly strong,
        # compact response spot. The focus mask itself never creates a bead.
        "log_threshold": 0.020,
        "min_raw_peak": 4.5,
        "min_center_mean": 2.8,
        "min_center_minus_ring": 0.85,
        "min_center_ring_ratio": 1.25,
        "min_display_value": 34,
        "min_hessian_ratio": 0.13,
        "max_support_axis_ratio": 3.6,
        "max_opposite_line_pairs": 1,
    },
}

# Local validation geometry.
PEAK_SNAP_RADIUS_PX = 2
CENTER_RADIUS_FACTOR = 0.70
RING_INNER_FACTOR = 1.25
RING_OUTER_FACTOR = 2.15
SUPPORT_THRESHOLD_FRACTION = 0.48
SUPPORT_PATCH_RADIUS_FACTOR = 2.5
DIRECTION_SAMPLE_RADIUS_FACTOR = 1.65
DIRECTION_CONTINUATION_FRACTION = 0.58

# Hard size limits on the local response support.
MIN_SUPPORT_AREA_PX = 2
MAX_SUPPORT_AREA_PX = 180

# Duplicate removal.
FINAL_DUPLICATE_DISTANCE_PX = 1.8
SOURCE_PRIORITY = {"focus": 3, "clutter": 2, "normal": 1}


# ============================================================
# OUTPUT SETTINGS
# ============================================================

CENTER_COLOR = (0, 0, 255)
CENTER_RADIUS = 2
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.9
FONT_THICKNESS = 2
BANNER_OPACITY = 0.75
JPEG_QUALITY = 95

USE_ESTIMATED_MASK_RADIUS = True
FIXED_MASK_RADIUS = 2
MIN_OUTPUT_MASK_RADIUS = 1
MAX_OUTPUT_MASK_RADIUS = 5

SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"
}

for folder in [
    RESULTS_DIR,
    OVERLAY_DIR,
    BEAD_MASK_DIR,
    DETECTION_TABLE_DIR,
    DEBUG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# TIMER
# ============================================================

class StageTimer:
    def __init__(self, image_name):
        self.image_name = image_name
        self.image_start = time.perf_counter()
        self.stage_start = self.image_start

    def reset_stage(self):
        self.stage_start = time.perf_counter()

    def report(self, stage_name):
        elapsed = time.perf_counter() - self.stage_start
        total = time.perf_counter() - self.image_start
        print(f"    {stage_name}: {elapsed:.2f} s (total {total:.2f} s)")
        self.stage_start = time.perf_counter()
        return elapsed

    def total(self):
        return time.perf_counter() - self.image_start


# ============================================================
# FILE HELPERS
# ============================================================

def ensure_odd(value):
    value = max(3, int(round(value)))
    return value + 1 if value % 2 == 0 else value


def read_image(path, flags=cv2.IMREAD_COLOR):
    path = Path(path)
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        if data.size == 0:
            return None
        return cv2.imdecode(data, flags)
    except Exception:
        return None


def write_image(path, image, params=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    params = [] if params is None else params
    success, encoded = cv2.imencode(path.suffix, image, params)
    if not success:
        raise IOError(f"Could not write image: {path}")
    encoded.tofile(str(path))


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(
        [
            p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
        ],
        key=lambda p: p.name.lower(),
    )


def remove_category_prefix(filename):
    return re.sub(
        r"^\s*n(?:45|52)\s*-\s*",
        "",
        filename,
        flags=re.IGNORECASE,
    )


# ============================================================
# MASK HELPERS
# ============================================================

def normalize_mask(mask):
    if mask is None:
        return None
    if mask.ndim == 3:
        mask = np.max(mask, axis=2)
    return np.where(mask > 0, 255, 0).astype(np.uint8)


def find_matching_mask(directory, image_stem, suffix):
    directory = Path(directory)
    if not directory.exists():
        return None
    expected = f"{image_stem}{suffix}".lower()
    for path in directory.iterdir():
        if (
            path.is_file()
            and path.suffix.lower() in SUPPORTED_EXTENSIONS
            and path.stem.lower() == expected
        ):
            return path
    return None


def load_mask(path, image_shape, default_value=0):
    height, width = image_shape[:2]
    if path is None:
        return np.full((height, width), default_value, dtype=np.uint8)
    mask = read_image(path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        return None
    mask = normalize_mask(mask)
    if mask.shape != (height, width):
        mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
        mask = normalize_mask(mask)
    return mask


def create_regions(image_shape, fov_path, artifact_path, clutter_path, focus_path):
    height, width = image_shape[:2]

    if USE_FOV_MASK:
        if fov_path is None:
            if SKIP_IF_FOV_MISSING:
                return None
            fov = np.full((height, width), 255, dtype=np.uint8)
        else:
            fov = load_mask(fov_path, image_shape, default_value=255)
    else:
        fov = np.full((height, width), 255, dtype=np.uint8)

    if fov is None:
        return None

    artifact = np.zeros((height, width), dtype=np.uint8)
    if USE_ARTIFACT_MASK:
        if artifact_path is not None:
            artifact = load_mask(artifact_path, image_shape, default_value=0)
            if artifact is None:
                return None
        elif not CONTINUE_IF_ARTIFACT_MISSING:
            return None

    clutter = np.zeros((height, width), dtype=np.uint8)
    if USE_CLUTTER_MASK:
        if clutter_path is not None:
            clutter = load_mask(clutter_path, image_shape, default_value=0)
            if clutter is None:
                return None
        elif not CONTINUE_IF_CLUTTER_MISSING:
            return None

    focus = np.zeros((height, width), dtype=np.uint8)
    if USE_FOCUS_MASK:
        if focus_path is not None:
            focus = load_mask(focus_path, image_shape, default_value=0)
            if focus is None:
                return None
        elif not CONTINUE_IF_FOCUS_MISSING:
            return None

    valid = np.where((fov > 0) & (artifact == 0), 255, 0).astype(np.uint8)

    if FOV_INNER_MARGIN_PX > 0:
        size = 2 * FOV_INNER_MARGIN_PX + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        valid = cv2.erode(valid, kernel, iterations=1)

    focus_region = np.where((focus > 0) & (valid > 0), 255, 0).astype(np.uint8)
    clutter_region = np.where(
        (clutter > 0) & (focus_region == 0) & (valid > 0),
        255,
        0,
    ).astype(np.uint8)
    normal_region = np.where(
        (valid > 0) & (focus_region == 0) & (clutter_region == 0),
        255,
        0,
    ).astype(np.uint8)

    return {
        "fov": fov,
        "artifact": artifact,
        "clutter": clutter,
        "focus": focus,
        "valid": valid,
        "normal_region": normal_region,
        "clutter_region": clutter_region,
        "focus_region": focus_region,
    }


# ============================================================
# RESPONSE IMAGE
# ============================================================

def create_gray_and_response(image_bgr, valid_region):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    smoothed = cv2.GaussianBlur(gray, (0, 0), GAUSSIAN_SIGMA)
    background = cv2.medianBlur(smoothed, ensure_odd(BACKGROUND_KERNEL))
    response = background.astype(np.float32) - smoothed.astype(np.float32)
    response = np.maximum(response, 0.0)
    response[valid_region == 0] = 0.0
    return gray, smoothed, background, response


def response_to_display(response, valid_region):
    values = response[(valid_region > 0) & (response > 0)]
    if values.size == 0:
        return np.zeros(response.shape, dtype=np.uint8)
    high = float(np.percentile(values, RESPONSE_DISPLAY_HIGH_PERCENTILE))
    high = max(high, 1.0)
    output = np.clip(response / high, 0.0, 1.0) * 255.0
    output[valid_region == 0] = 0
    return output.astype(np.uint8)


# ============================================================
# RESPONSE-SPOT MEASUREMENTS
# ============================================================

def disk_and_ring_values(image, region_mask, x, y, radius):
    h, w = image.shape
    outer = max(4, int(np.ceil(radius * RING_OUTER_FACTOR)))
    x0, x1 = max(0, x - outer), min(w, x + outer + 1)
    y0, y1 = max(0, y - outer), min(h, y + outer + 1)

    patch = image[y0:y1, x0:x1]
    allowed = region_mask[y0:y1, x0:x1] > 0
    yy, xx = np.ogrid[:patch.shape[0], :patch.shape[1]]
    lx, ly = x - x0, y - y0
    distance = np.sqrt((xx - lx) ** 2 + (yy - ly) ** 2)

    center = distance <= max(1.0, radius * CENTER_RADIUS_FACTOR)
    ring = (
        (distance >= radius * RING_INNER_FACTOR)
        & (distance <= radius * RING_OUTER_FACTOR)
    )

    center_values = patch[center & allowed]
    ring_values = patch[ring & allowed]
    if center_values.size == 0 or ring_values.size == 0:
        return None

    return {
        "center_mean": float(np.mean(center_values)),
        "center_median": float(np.median(center_values)),
        "ring_mean": float(np.mean(ring_values)),
        "ring_median": float(np.median(ring_values)),
    }


def snap_to_response_peak(response, region_mask, x, y, radius=PEAK_SNAP_RADIUS_PX):
    h, w = response.shape
    x0, x1 = max(0, x - radius), min(w, x + radius + 1)
    y0, y1 = max(0, y - radius), min(h, y + radius + 1)
    patch = response[y0:y1, x0:x1].copy()
    allowed = region_mask[y0:y1, x0:x1] > 0
    patch[~allowed] = -1.0
    if patch.size == 0 or np.max(patch) < 0:
        return x, y
    py, px = np.unravel_index(np.argmax(patch), patch.shape)
    return x0 + int(px), y0 + int(py)


def hessian_blob_ratio(dxx, dyy, dxy, x, y):
    # Bright compact spots have two negative curvatures of comparable size.
    a = float(dxx[y, x])
    b = float(dxy[y, x])
    c = float(dyy[y, x])
    trace = a + c
    disc = max(0.0, (a - c) ** 2 + 4.0 * b * b)
    root = np.sqrt(disc)
    eig1 = 0.5 * (trace - root)
    eig2 = 0.5 * (trace + root)

    # At a bright maximum, both should be negative. Reject saddles/ridges.
    if eig1 >= 0 or eig2 >= 0:
        return 0.0, eig1, eig2

    magnitude1 = abs(eig1)
    magnitude2 = abs(eig2)
    ratio = min(magnitude1, magnitude2) / max(magnitude1, magnitude2, 1e-6)
    return float(ratio), eig1, eig2


def local_support_shape(response, region_mask, x, y, radius, peak_value):
    h, w = response.shape
    patch_radius = max(4, int(np.ceil(radius * SUPPORT_PATCH_RADIUS_FACTOR)))
    x0, x1 = max(0, x - patch_radius), min(w, x + patch_radius + 1)
    y0, y1 = max(0, y - patch_radius), min(h, y + patch_radius + 1)

    patch = response[y0:y1, x0:x1]
    allowed = region_mask[y0:y1, x0:x1] > 0
    threshold = max(1.0, float(peak_value) * SUPPORT_THRESHOLD_FRACTION)
    binary = ((patch >= threshold) & allowed).astype(np.uint8)

    lx, ly = x - x0, y - y0
    if not (0 <= ly < binary.shape[0] and 0 <= lx < binary.shape[1]):
        return None
    if binary[ly, lx] == 0:
        return None

    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    label_id = int(labels[ly, lx])
    if label_id <= 0 or label_id >= count:
        return None

    area = int(stats[label_id, cv2.CC_STAT_AREA])
    component = (labels == label_id).astype(np.uint8)
    points = np.column_stack(np.nonzero(component))

    if points.shape[0] < 2:
        axis_ratio = 1.0
    else:
        coords = points[:, [1, 0]].astype(np.float32)
        covariance = np.cov(coords, rowvar=False)
        eigenvalues = np.linalg.eigvalsh(covariance)
        eigenvalues = np.maximum(eigenvalues, 1e-6)
        axis_ratio = float(np.sqrt(eigenvalues[-1] / eigenvalues[0]))

    return {
        "area": area,
        "axis_ratio": axis_ratio,
        "threshold": threshold,
    }


def count_opposite_line_pairs(response, x, y, radius, peak_value):
    h, w = response.shape
    sample_radius = max(2, int(round(radius * DIRECTION_SAMPLE_RADIUS_FACTOR)))
    continuation_threshold = peak_value * DIRECTION_CONTINUATION_FRACTION

    directions = [
        (1, 0),
        (1, 1),
        (0, 1),
        (-1, 1),
    ]

    pair_count = 0
    values = []

    for dx, dy in directions:
        x1 = int(np.clip(x + dx * sample_radius, 0, w - 1))
        y1 = int(np.clip(y + dy * sample_radius, 0, h - 1))
        x2 = int(np.clip(x - dx * sample_radius, 0, w - 1))
        y2 = int(np.clip(y - dy * sample_radius, 0, h - 1))
        v1 = float(response[y1, x1])
        v2 = float(response[y2, x2])
        values.append((v1, v2))
        if v1 >= continuation_threshold and v2 >= continuation_threshold:
            pair_count += 1

    return pair_count, values


# ============================================================
# RESPONSE-ONLY DETECTOR
# ============================================================

def detect_response_spots(response, response_display, region_mask, source_name):
    settings = REGION_SETTINGS[source_name]
    if np.count_nonzero(region_mask) == 0:
        return [], []

    response_input = (response / 255.0).astype(np.float32)
    response_input *= (region_mask > 0)

    blobs = blob_log(
        response_input,
        min_sigma=DETECT_MIN_SIGMA,
        max_sigma=DETECT_MAX_SIGMA,
        num_sigma=DETECT_NUM_SIGMA,
        threshold=settings["log_threshold"],
        overlap=DETECT_OVERLAP,
        exclude_border=False,
    )

    response_smoothed = cv2.GaussianBlur(response, (0, 0), 0.8)
    dxx = cv2.Sobel(response_smoothed, cv2.CV_32F, 2, 0, ksize=3)
    dyy = cv2.Sobel(response_smoothed, cv2.CV_32F, 0, 2, ksize=3)
    dxy = cv2.Sobel(response_smoothed, cv2.CV_32F, 1, 1, ksize=3)
    h, w = response.shape
    accepted = []
    rejected = []

    for y_float, x_float, sigma in blobs:
        x = int(round(x_float))
        y = int(round(y_float))
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        if region_mask[y, x] == 0:
            continue

        radius = float(np.sqrt(2.0) * sigma)
        diameter = 2.0 * radius
        if diameter < MIN_BEAD_DIAMETER_PX * 0.60:
            continue
        if diameter > MAX_BEAD_DIAMETER_PX * 1.40:
            continue

        x, y = snap_to_response_peak(response, region_mask, x, y)
        peak_value = float(response[y, x])
        display_value = int(response_display[y, x])

        record = {
            "x": x,
            "y": y,
            "radius_px": radius,
            "diameter_px": diameter,
            "diameter_um": diameter * MICRONS_PER_PIXEL,
            "response": peak_value,
            "response_display": display_value,
            "source": source_name,
        }

        if peak_value < settings["min_raw_peak"]:
            record["rejection_reason"] = "weak_raw_peak"
            rejected.append(record)
            continue

        if display_value < settings["min_display_value"]:
            record["rejection_reason"] = "not_visibly_bright"
            rejected.append(record)
            continue

        measurements = disk_and_ring_values(response, region_mask, x, y, radius)
        if measurements is None:
            record["rejection_reason"] = "invalid_center_ring"
            rejected.append(record)
            continue

        center_mean = measurements["center_mean"]
        ring_mean = measurements["ring_mean"]
        center_minus_ring = center_mean - ring_mean
        center_ring_ratio = center_mean / max(ring_mean, 0.25)

        record.update({
            "center_response_mean": center_mean,
            "ring_response_mean": ring_mean,
            "center_minus_ring": center_minus_ring,
            "center_ring_ratio": center_ring_ratio,
        })

        if center_mean < settings["min_center_mean"]:
            record["rejection_reason"] = "weak_center_mean"
            rejected.append(record)
            continue

        # Use both an absolute and relative separation requirement.
        if (
            center_minus_ring < settings["min_center_minus_ring"]
            or center_ring_ratio < settings["min_center_ring_ratio"]
        ):
            record["rejection_reason"] = "not_localized_above_ring"
            rejected.append(record)
            continue

        hessian_ratio, eig1, eig2 = hessian_blob_ratio(dxx, dyy, dxy, x, y)
        record.update({
            "hessian_blob_ratio": hessian_ratio,
            "hessian_eigenvalue_1": eig1,
            "hessian_eigenvalue_2": eig2,
        })

        if hessian_ratio < settings["min_hessian_ratio"]:
            record["rejection_reason"] = "ridge_like_hessian"
            rejected.append(record)
            continue

        support = local_support_shape(response, region_mask, x, y, radius, peak_value)
        if support is None:
            record["rejection_reason"] = "invalid_support"
            rejected.append(record)
            continue

        record.update({
            "support_area_px": support["area"],
            "support_axis_ratio": support["axis_ratio"],
            "support_threshold": support["threshold"],
        })

        if support["area"] < MIN_SUPPORT_AREA_PX:
            record["rejection_reason"] = "support_too_small"
            rejected.append(record)
            continue

        if support["area"] > MAX_SUPPORT_AREA_PX:
            record["rejection_reason"] = "support_too_large"
            rejected.append(record)
            continue

        if support["axis_ratio"] > settings["max_support_axis_ratio"]:
            record["rejection_reason"] = "elongated_support"
            rejected.append(record)
            continue

        opposite_pairs, direction_values = count_opposite_line_pairs(
            response,
            x,
            y,
            radius,
            peak_value,
        )
        record["opposite_line_pairs"] = opposite_pairs

        if opposite_pairs > settings["max_opposite_line_pairs"]:
            record["rejection_reason"] = "continuous_line_response"
            rejected.append(record)
            continue

        # Final score prioritizes genuinely strong, compact response spots.
        record["score"] = (
            peak_value
            + 2.0 * center_minus_ring
            + 10.0 * hessian_ratio
            - 0.25 * support["axis_ratio"]
        )
        record["rejection_reason"] = ""
        accepted.append(record)

    return accepted, rejected


# ============================================================
# DUPLICATE MERGING
# ============================================================

def merge_detections_kdtree(detections, minimum_distance):
    if not detections:
        return []

    detections = sorted(
        detections,
        key=lambda d: (
            SOURCE_PRIORITY.get(d.get("source", ""), 0),
            float(d.get("score", 0.0)),
        ),
        reverse=True,
    )

    coordinates = np.array([[d["x"], d["y"]] for d in detections], dtype=np.float32)
    tree = cKDTree(coordinates)
    removed = np.zeros(len(detections), dtype=bool)
    accepted = []

    for index, detection in enumerate(detections):
        if removed[index]:
            continue
        accepted.append(detection)
        for neighbor in tree.query_ball_point(coordinates[index], r=minimum_distance):
            if neighbor > index:
                removed[neighbor] = True

    accepted = sorted(accepted, key=lambda d: (d["y"], d["x"]))
    for bead_id, detection in enumerate(accepted, start=1):
        detection["id"] = bead_id
    return accepted


# ============================================================
# OUTPUT GENERATION
# ============================================================

def create_bead_mask(image_shape, detections, valid_region):
    h, w = image_shape[:2]
    output = np.zeros((h, w), dtype=np.uint8)
    for detection in detections:
        if USE_ESTIMATED_MASK_RADIUS:
            radius = int(round(detection["radius_px"]))
        else:
            radius = FIXED_MASK_RADIUS
        radius = int(np.clip(radius, MIN_OUTPUT_MASK_RADIUS, MAX_OUTPUT_MASK_RADIUS))
        cv2.circle(output, (int(detection["x"]), int(detection["y"])), radius, 255, -1)
    output[valid_region == 0] = 0
    return output


def add_count_banner(image, count):
    output = image.copy()
    text = f"Total count = {count}"
    (text_width, text_height), baseline = cv2.getTextSize(
        text, FONT, FONT_SCALE, FONT_THICKNESS
    )
    padding_x, padding_y = 15, 12
    banner_width = text_width + 2 * padding_x
    banner_height = text_height + baseline + 2 * padding_y
    layer = output.copy()
    cv2.rectangle(layer, (0, 0), (banner_width, banner_height), (255, 255, 255), -1)
    cv2.addWeighted(layer, BANNER_OPACITY, output, 1.0 - BANNER_OPACITY, 0, output)
    cv2.putText(
        output,
        text,
        (padding_x, padding_y + text_height),
        FONT,
        FONT_SCALE,
        (0, 0, 0),
        FONT_THICKNESS,
        cv2.LINE_AA,
    )
    return output


def create_overlay(image_bgr, detections):
    overlay = image_bgr.copy()
    for detection in detections:
        cv2.circle(
            overlay,
            (int(detection["x"]), int(detection["y"])),
            CENTER_RADIUS,
            CENTER_COLOR,
            -1,
            cv2.LINE_AA,
        )
    return add_count_banner(overlay, len(detections))


def create_source_overlay(image_bgr, detections):
    colors = {
        "normal": (255, 0, 0),
        "clutter": (0, 255, 0),
        "focus": (255, 0, 255),
    }
    overlay = image_bgr.copy()
    for detection in detections:
        color = colors.get(detection.get("source", ""), (255, 255, 255))
        cv2.circle(
            overlay,
            (int(detection["x"]), int(detection["y"])),
            3,
            color,
            1,
            cv2.LINE_AA,
        )
    return overlay


def create_rejected_overlay(image_bgr, rejected):
    overlay = image_bgr.copy()
    for detection in rejected:
        cv2.circle(
            overlay,
            (int(detection["x"]), int(detection["y"])),
            1,
            (255, 255, 0),
            -1,
            cv2.LINE_AA,
        )
    return overlay


# ============================================================
# DEBUG OUTPUT
# ============================================================

def save_debug_outputs(
    image_path,
    category,
    image_bgr,
    regions,
    background,
    response_display,
    strong_response_mask,
    normal_detections,
    clutter_detections,
    focus_detections,
    rejected,
    final_detections,
    bead_mask,
    final_overlay,
):
    output_folder = DEBUG_DIR / category / image_path.stem
    output_folder.mkdir(parents=True, exist_ok=True)

    images = {
        "01_original.jpg": image_bgr,
        "02_fov.png": regions["fov"],
        "03_artifact.png": regions["artifact"],
        "04_clutter.png": regions["clutter_region"],
        "05_focus.png": regions["focus_region"],
        "06_normal_region.png": regions["normal_region"],
        "07_background.png": background,
        "08_response_used_for_detection.png": response_display,
        "09_strong_response_mask.png": strong_response_mask,
        "10_normal_detections.jpg": create_source_overlay(image_bgr, normal_detections),
        "11_clutter_detections.jpg": create_source_overlay(image_bgr, clutter_detections),
        "12_focus_detections.jpg": create_source_overlay(image_bgr, focus_detections),
        "13_rejected_response_candidates.jpg": create_rejected_overlay(image_bgr, rejected),
        "14_accepted_sources.jpg": create_source_overlay(image_bgr, final_detections),
        "15_bead_mask.png": bead_mask,
        "16_final_overlay.jpg": final_overlay,
    }

    for filename, image in images.items():
        path = output_folder / filename
        if path.suffix.lower() == ".jpg":
            write_image(path, image, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        else:
            write_image(path, image)


def show_debug(image_bgr, response_display, strong_response_mask, final_overlay):
    if not SHOW_DEBUG_IMAGES:
        return
    items = [
        ("Original", cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), None),
        ("Response used for detection", response_display, "gray"),
        ("Strong response mask", strong_response_mask, "gray"),
        ("Final overlay", cv2.cvtColor(final_overlay, cv2.COLOR_BGR2RGB), None),
    ]
    for title, image, cmap in items:
        plt.figure(figsize=(14, 9))
        plt.imshow(image, cmap=cmap)
        plt.title(title)
        plt.axis("off")
        plt.show()


# ============================================================
# PROCESS ONE IMAGE
# ============================================================

def process_image(image_path, category):
    image_path = Path(image_path)
    timer = StageTimer(image_path.name)

    print("\n" + "=" * 78)
    print(f"Category: {category}")
    print(f"Image: {image_path.name}")
    print("=" * 78)

    result = {
        "Filename": remove_category_prefix(image_path.name),
        "Category": category,
        "Final count": np.nan,
        "Normal count": np.nan,
        "Clutter count": np.nan,
        "Focus count": np.nan,
        "Rejected candidates": np.nan,
        "Processing time (s)": np.nan,
        "Status": "",
    }

    image_bgr = read_image(image_path, cv2.IMREAD_COLOR)
    timer.report("Image loading")
    if image_bgr is None:
        result["Status"] = "image_read_failed"
        return result

    fov_path = find_matching_mask(FOV_MASK_DIR, image_path.stem, FOV_MASK_SUFFIX)
    artifact_path = find_matching_mask(
        ARTIFACT_MASK_DIR, image_path.stem, ARTIFACT_MASK_SUFFIX
    )
    clutter_path = find_matching_mask(
        CLUTTER_MASK_DIR, image_path.stem, CLUTTER_MASK_SUFFIX
    )
    focus_path = find_matching_mask(FOCUS_MASK_DIR, image_path.stem, FOCUS_MASK_SUFFIX)

    print(f"    FOV mask: {fov_path.name if fov_path else 'none'}")
    print(f"    Artifact mask: {artifact_path.name if artifact_path else 'none'}")
    print(f"    Clutter mask: {clutter_path.name if clutter_path else 'none'}")
    print(f"    Focus mask: {focus_path.name if focus_path else 'none'}")

    regions = create_regions(
        image_bgr.shape,
        fov_path,
        artifact_path,
        clutter_path,
        focus_path,
    )
    timer.report("Mask loading and region creation")
    if regions is None:
        result["Status"] = "region_creation_failed"
        return result
    if np.count_nonzero(regions["valid"]) == 0:
        result["Status"] = "empty_valid_region"
        return result

    gray, smoothed, background, response = create_gray_and_response(
        image_bgr, regions["valid"]
    )
    response_display = response_to_display(response, regions["valid"])
    timer.report("Response creation")

    # This mask is only for diagnostics. It shows which visibly strong response
    # pixels are even eligible for the detector's hard brightness checks.
    minimum_display = min(s["min_display_value"] for s in REGION_SETTINGS.values())
    strong_response_mask = np.where(
        (response_display >= minimum_display) & (regions["valid"] > 0),
        255,
        0,
    ).astype(np.uint8)

    normal_detections, normal_rejected = detect_response_spots(
        response,
        response_display,
        regions["normal_region"],
        "normal",
    )
    timer.report("Normal response-spot detector")

    clutter_detections, clutter_rejected = detect_response_spots(
        response,
        response_display,
        regions["clutter_region"],
        "clutter",
    )
    timer.report("Clutter response-spot detector")

    focus_detections, focus_rejected = detect_response_spots(
        response,
        response_display,
        regions["focus_region"],
        "focus",
    )
    timer.report("Focus response-spot detector")

    all_detections = normal_detections + clutter_detections + focus_detections
    all_rejected = normal_rejected + clutter_rejected + focus_rejected
    final_detections = merge_detections_kdtree(
        all_detections,
        FINAL_DUPLICATE_DISTANCE_PX,
    )
    timer.report("Duplicate merging")

    bead_mask = create_bead_mask(image_bgr.shape, final_detections, regions["valid"])
    final_overlay = create_overlay(image_bgr, final_detections)

    category_overlay_dir = OVERLAY_DIR / category
    category_mask_dir = BEAD_MASK_DIR / category
    category_table_dir = DETECTION_TABLE_DIR / category
    for folder in [category_overlay_dir, category_mask_dir, category_table_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    write_image(
        category_overlay_dir / f"{image_path.stem}_overlay.jpg",
        final_overlay,
        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
    )
    write_image(
        category_mask_dir / f"{image_path.stem}_bead_mask.png",
        bead_mask,
    )
    pd.DataFrame(final_detections).to_csv(
        category_table_dir / f"{image_path.stem}_detections.csv",
        index=False,
    )
    pd.DataFrame(all_rejected).to_csv(
        category_table_dir / f"{image_path.stem}_rejected.csv",
        index=False,
    )
    timer.report("Saving standard outputs")

    if DEBUG_MODE:
        if SAVE_DEBUG_IMAGES:
            save_debug_outputs(
                image_path,
                category,
                image_bgr,
                regions,
                background,
                response_display,
                strong_response_mask,
                normal_detections,
                clutter_detections,
                focus_detections,
                all_rejected,
                final_detections,
                bead_mask,
                final_overlay,
            )
        timer.report("Saving debug outputs")
        show_debug(image_bgr, response_display, strong_response_mask, final_overlay)

    final_normal_count = sum(d["source"] == "normal" for d in final_detections)
    final_clutter_count = sum(d["source"] == "clutter" for d in final_detections)
    final_focus_count = sum(d["source"] == "focus" for d in final_detections)
    total_time = timer.total()

    result.update({
        "Final count": len(final_detections),
        "Normal count": final_normal_count,
        "Clutter count": final_clutter_count,
        "Focus count": final_focus_count,
        "Rejected candidates": len(all_rejected),
        "Processing time (s)": round(total_time, 3),
        "Status": "processed",
    })

    print("-" * 78)
    print(f"Final normal count: {final_normal_count}")
    print(f"Final clutter count: {final_clutter_count}")
    print(f"Final focus count: {final_focus_count}")
    print(f"Rejected response candidates: {len(all_rejected)}")
    print(f"FINAL COUNT: {len(final_detections)}")
    print(f"TOTAL PROCESSING TIME: {total_time:.2f} seconds")
    return result


# ============================================================
# BUILD IMAGE LIST
# ============================================================

images_to_process = []

if DEBUG_MODE:
    if not DEBUG_IMAGE_PATH.exists():
        raise FileNotFoundError(
            "Debug image does not exist:\n"
            f"{DEBUG_IMAGE_PATH.resolve()}"
        )

    parent_name = DEBUG_IMAGE_PATH.parent.name.lower()
    filename_lower = DEBUG_IMAGE_PATH.name.lower()
    if "n45" in parent_name or filename_lower.startswith("n45"):
        category = "n45"
    elif "n52" in parent_name or filename_lower.startswith("n52"):
        category = "n52"
    else:
        raise ValueError(
            "Could not determine whether the debug image belongs to n45 or n52."
        )

    images_to_process.append((category, DEBUG_IMAGE_PATH))
    print("DEBUG MODE ENABLED")
    print("Only one image will be processed:")
    print(DEBUG_IMAGE_PATH)
else:
    for category, directory in DATASETS.items():
        images = list_images(directory)
        print(f"{category}: {len(images)} images found")
        for image_path in images:
            images_to_process.append((category, image_path))

    if not images_to_process:
        raise FileNotFoundError("No images were found in images_n45 or images_n52.")
    print(f"Total images to process: {len(images_to_process)}")


# ============================================================
# RUN PROCESSING
# ============================================================

batch_start = time.perf_counter()
results = []

for image_index, (category, image_path) in enumerate(images_to_process, start=1):
    print(f"\nIMAGE {image_index} OF {len(images_to_process)}")
    try:
        image_result = process_image(image_path, category)
    except KeyboardInterrupt:
        print("\nProcessing interrupted by user.")
        raise
    except Exception as error:
        print(f"\nERROR processing {image_path.name}:")
        print(error)
        traceback.print_exc()
        image_result = {
            "Filename": remove_category_prefix(image_path.name),
            "Category": category,
            "Final count": np.nan,
            "Normal count": np.nan,
            "Clutter count": np.nan,
            "Focus count": np.nan,
            "Rejected candidates": np.nan,
            "Processing time (s)": np.nan,
            "Status": f"error: {type(error).__name__}",
        }
    results.append(image_result)
    print(f"\nCompleted image {image_index} of {len(images_to_process)}")


# ============================================================
# SAVE FINAL EXCEL
# ============================================================

results_dataframe = pd.DataFrame(results)
if not results_dataframe.empty:
    category_order = {"n45": 0, "n52": 1}
    results_dataframe["_category_order"] = (
        results_dataframe["Category"].map(category_order).fillna(99)
    )
    results_dataframe = (
        results_dataframe
        .sort_values(["_category_order", "Filename"])
        .drop(columns=["_category_order"])
        .reset_index(drop=True)
    )

final_counts_dataframe = results_dataframe[
    ["Filename", "Category", "Final count"]
].copy()

with pd.ExcelWriter(FINAL_EXCEL_PATH, engine="openpyxl") as writer:
    final_counts_dataframe.to_excel(writer, sheet_name="Final counts", index=False)
    results_dataframe.to_excel(writer, sheet_name="Processing details", index=False)
    workbook = writer.book
    for sheet_name in ["Final counts", "Processing details"]:
        worksheet = workbook[sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter
            maximum_length = max(
                len(str(cell.value) if cell.value is not None else "")
                for cell in column_cells
            )
            worksheet.column_dimensions[column_letter].width = min(
                maximum_length + 2,
                40,
            )


# ============================================================
# FINAL SUMMARY
# ============================================================

batch_time = time.perf_counter() - batch_start
print("\n" + "=" * 78)
print("PROCESSING COMPLETE")
print("=" * 78)
print(final_counts_dataframe.to_string(index=False))
print(f"\nImages attempted: {len(images_to_process)}")
print(f"Total batch time: {batch_time:.2f} seconds")
if len(images_to_process) > 0:
    print(f"Average time per image: {batch_time / len(images_to_process):.2f} seconds")
print(f"\nExcel file saved to:\n{FINAL_EXCEL_PATH.resolve()}")

final_counts_dataframe

Scale: 3.7037 µm/pixel
Expected bead diameter: 2.70–9.45 pixels
n45: 44 images found
n52: 40 images found
Total images to process: 84

IMAGE 1 OF 84

Category: n45
Image: n45 - S1-1st wash.jpg
    Image loading: 0.07 s (total 0.07 s)
    FOV mask: n45 - S1-1st wash_fov_mask.png
    Artifact mask: none
    Clutter mask: n45 - S1-1st wash_clutter_mask.png
    Focus mask: n45 - S1-1st wash_focus_mask.png
    Mask loading and region creation: 0.39 s (total 0.45 s)
    Response creation: 0.31 s (total 0.76 s)
    Normal response-spot detector: 6.51 s (total 7.28 s)
    Clutter response-spot detector: 0.00 s (total 7.28 s)
    Focus response-spot detector: 5.85 s (total 13.13 s)
    Duplicate merging: 0.02 s (total 13.15 s)
    Saving standard outputs: 0.15 s (total 13.30 s)
------------------------------------------------------------------------------
Final normal count: 488
Final clutter count: 0
Final focus count: 499
Rejected response candidates: 61
FINAL COUNT: 987
TOTAL PROCESSING TIME

,Filename,Category,Final count
0,S1-1st wash.jpg,n45,987
1,S1-2nd wash.jpg,n45,672
2,S1-3rd wash.jpg,n45,763
3,S1-b4 Magnet.jpg,n45,1086
4,S10-1st wash.jpg,n45,560
...,...,...,...
79,S8-before megnet.jpg,n52,786
80,S9-1st wash.jpg,n52,1202
81,S9-2nd Wash.jpg,n52,1033
82,S9-3rd wash.jpg,n52,1031


## Segmented

In [6]:
# ============================================================
# SEGMENTED MAGNETIC BEAD COUNTER - SPECIAL CLUTTER ROUTE
#
# Core rule:
#   1. Build the same local-dark response used by the earlier code.
#   2. Detect beads ONLY from strong, compact spots in that response.
#   3. Use the original image only for the final overlay.
#   4. FOV/artifact masks restrict valid pixels.
#   5. Focus/clutter masks only select sensitivity. They are NOT bead masks.
#
# INPUT IMAGE FOLDERS
#     ./images_segmented/
#
# MASK FOLDERS
#     ./annotate_segmented/labeled_images/fov/
#     ./annotate_segmented/labeled_images/artifact/
#     ./annotate_segmented/labeled_images/clutter/
#     ./annotate_segmented/labeled_images/focus/
#
# OUTPUT
#     ./results_segmented_special_clutter/
# ============================================================

from pathlib import Path
import re
import time
import traceback

import cv2
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from skimage.feature import blob_log, peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(".")

IMAGES_DIR = PROJECT_DIR / "images_segmented"

LABEL_DIR = PROJECT_DIR / "annotate_segmented" / "labeled_images"
FOV_MASK_DIR = LABEL_DIR / "fov"
ARTIFACT_MASK_DIR = LABEL_DIR / "artifact"
CLUTTER_MASK_DIR = LABEL_DIR / "clutter"
FOCUS_MASK_DIR = LABEL_DIR / "focus"

RESULTS_DIR = PROJECT_DIR / "results_segmented_special_clutter"
OVERLAY_DIR = RESULTS_DIR / "overlays"
BEAD_MASK_DIR = RESULTS_DIR / "bead_masks"
DETECTION_TABLE_DIR = RESULTS_DIR / "detection_tables"
DEBUG_DIR = RESULTS_DIR / "debug"
FINAL_EXCEL_PATH = RESULTS_DIR / "final_counts.xlsx"


# ============================================================
# DEBUG OR BATCH MODE
# ============================================================

DEBUG_MODE = False
DEBUG_IMAGE_PATH = Path("./images_segmented/S1-3rd Wash_section_01.png")
SHOW_DEBUG_IMAGES = True
SAVE_DEBUG_IMAGES = True

# ============================================================
# MASK SETTINGS
# ============================================================

USE_FOV_MASK = True
USE_ARTIFACT_MASK = True
USE_CLUTTER_MASK = True
USE_FOCUS_MASK = True

SKIP_IF_FOV_MISSING = True
CONTINUE_IF_ARTIFACT_MISSING = True
CONTINUE_IF_CLUTTER_MISSING = True
CONTINUE_IF_FOCUS_MISSING = True

FOV_MASK_SUFFIX = "_fov_mask"
ARTIFACT_MASK_SUFFIX = "_artifact_mask"
CLUTTER_MASK_SUFFIX = "_clutter_mask"
FOCUS_MASK_SUFFIX = "_focus_mask"

FOV_INNER_MARGIN_PX = 1

# Expand only the manually marked clutter region slightly so beads touching
# the annotation boundary use the special clutter detector.
CLUTTER_MASK_DILATION_RADIUS_PX = 2


# ============================================================
# PHYSICAL SCALE
# ============================================================

SCALE_PIXELS = 135.0
SCALE_MICRONS = 500.0
MICRONS_PER_PIXEL = SCALE_MICRONS / SCALE_PIXELS
PIXELS_PER_MICRON = SCALE_PIXELS / SCALE_MICRONS

MIN_BEAD_DIAMETER_UM = 10.0
MAX_BEAD_DIAMETER_UM = 35.0
MIN_BEAD_DIAMETER_PX = MIN_BEAD_DIAMETER_UM * PIXELS_PER_MICRON
MAX_BEAD_DIAMETER_PX = MAX_BEAD_DIAMETER_UM * PIXELS_PER_MICRON
MIN_BEAD_RADIUS_PX = MIN_BEAD_DIAMETER_PX / 2.0
MAX_BEAD_RADIUS_PX = MAX_BEAD_DIAMETER_PX / 2.0
TYPICAL_BEAD_RADIUS_PX = max(
    1.5,
    (MIN_BEAD_RADIUS_PX + MAX_BEAD_RADIUS_PX) / 2.0,
)

print(f"Scale: {MICRONS_PER_PIXEL:.4f} µm/pixel")
print(
    "Expected bead diameter: "
    f"{MIN_BEAD_DIAMETER_PX:.2f}–{MAX_BEAD_DIAMETER_PX:.2f} pixels"
)


# ============================================================
# RESPONSE CREATION
# ============================================================

GAUSSIAN_SIGMA = 0.55
BACKGROUND_KERNEL = 31

# Display scaling only. Detection uses the raw float response.
RESPONSE_DISPLAY_HIGH_PERCENTILE = 99.80


# ============================================================
# RESPONSE-ONLY DETECTOR SETTINGS
# ============================================================

# LoG size range, derived from expected bead size.
DETECT_MIN_SIGMA = max(
    0.50,
    MIN_BEAD_DIAMETER_PX / (2.0 * np.sqrt(2.0)),
)
DETECT_MAX_SIGMA = max(
    DETECT_MIN_SIGMA + 0.20,
    MAX_BEAD_DIAMETER_PX / (2.0 * np.sqrt(2.0)),
)
DETECT_NUM_SIGMA = 10
DETECT_OVERLAP = 0.80

# Thresholds are expressed on response / 255, exactly like the successful
# normal LoG detector in the original code. These values prevent faint
# response speckle from becoming candidates.
REGION_SETTINGS = {
    "normal": {
        "use_peak_supplement": False,
        "log_threshold": 0.026,
        "min_raw_peak": 5.5,
        "min_center_mean": 3.5,
        "min_center_minus_ring": 1.15,
        "min_center_ring_ratio": 1.35,
        "min_display_value": 42,
        "min_hessian_ratio": 0.16,
        "max_support_axis_ratio": 3.2,
        "max_opposite_line_pairs": 1,
    },
    "clutter": {
        # Dense clutter needs extra sensitivity because touching beads can
        # suppress one another in the LoG response. A second local-maximum
        # candidate pass is enabled only inside the clutter mask.
        "log_threshold": 0.014,
        "min_raw_peak": 2.8,
        "min_center_mean": 1.8,
        "min_center_minus_ring": 0.35,
        "min_center_ring_ratio": 1.10,
        "min_display_value": 20,
        "min_hessian_ratio": 0.045,
        "max_support_axis_ratio": 5.5,
        "max_opposite_line_pairs": 2,
        "use_peak_supplement": True,
        "peak_threshold_raw": 2.5,
        "peak_min_distance": 1,
        "peak_max_candidates": 12000,
    },
    "focus": {
        "use_peak_supplement": False,
        # Focus is more sensitive, but still requires a visibly strong,
        # compact response spot. The focus mask itself never creates a bead.
        "log_threshold": 0.020,
        "min_raw_peak": 4.5,
        "min_center_mean": 2.8,
        "min_center_minus_ring": 0.85,
        "min_center_ring_ratio": 1.25,
        "min_display_value": 34,
        "min_hessian_ratio": 0.13,
        "max_support_axis_ratio": 3.6,
        "max_opposite_line_pairs": 1,
    },
}

# ============================================================
# SPECIAL CLUTTER DETECTOR
#
# This is the dedicated clutter method transplanted from the older pipeline.
# It runs only inside manually marked clutter pixels. Normal and focus
# processing remain unchanged.
# ============================================================

CLUTTER_RESPONSE_SIGMA = 0.35
CLUTTER_FOREGROUND_RESPONSE_THRESHOLD = 4.0
CLUTTER_PEAK_MIN_RESPONSE = 9.0
CLUTTER_PEAK_MIN_DISTANCE_PX = 1
CLUTTER_PEAK_THRESHOLD_REL = 0.025
CLUTTER_MIN_PEAK_PROMINENCE = 0.10
CLUTTER_MIN_LOCAL_CONTRAST = -1.0
CLUTTER_MIN_RESPONSE_DIFFERENCE = -1.0
CLUTTER_MAX_CENTER_GRAY = 235
CLUTTER_CLOSE_RADIUS_PX = 1
CLUTTER_MIN_FOREGROUND_COMPONENT_AREA_PX = 2
CLUTTER_MIN_WATERSHED_AREA_PX = 1
CLUTTER_MAX_WATERSHED_AREA_PX = 150
CLUTTER_MAX_PEAKS_PER_COMPONENT = 10000

# Local validation geometry.
PEAK_SNAP_RADIUS_PX = 2
CENTER_RADIUS_FACTOR = 0.70
RING_INNER_FACTOR = 1.25
RING_OUTER_FACTOR = 2.15
SUPPORT_THRESHOLD_FRACTION = 0.48
SUPPORT_PATCH_RADIUS_FACTOR = 2.5
DIRECTION_SAMPLE_RADIUS_FACTOR = 1.65
DIRECTION_CONTINUATION_FRACTION = 0.58

# Hard size limits on the local response support.
MIN_SUPPORT_AREA_PX = 2
MAX_SUPPORT_AREA_PX = 180

# Duplicate removal.
FINAL_DUPLICATE_DISTANCE_PX = 1.8
SOURCE_PRIORITY = {"focus": 3, "clutter": 2, "normal": 1}


# ============================================================
# OUTPUT SETTINGS
# ============================================================

CENTER_COLOR = (0, 0, 255)
CENTER_RADIUS = 2
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.9
FONT_THICKNESS = 2
BANNER_OPACITY = 0.75
JPEG_QUALITY = 95

USE_ESTIMATED_MASK_RADIUS = True
FIXED_MASK_RADIUS = 2
MIN_OUTPUT_MASK_RADIUS = 1
MAX_OUTPUT_MASK_RADIUS = 5

SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"
}

for folder in [
    RESULTS_DIR,
    OVERLAY_DIR,
    BEAD_MASK_DIR,
    DETECTION_TABLE_DIR,
    DEBUG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# TIMER
# ============================================================

class StageTimer:
    def __init__(self, image_name):
        self.image_name = image_name
        self.image_start = time.perf_counter()
        self.stage_start = self.image_start

    def reset_stage(self):
        self.stage_start = time.perf_counter()

    def report(self, stage_name):
        elapsed = time.perf_counter() - self.stage_start
        total = time.perf_counter() - self.image_start
        print(f"    {stage_name}: {elapsed:.2f} s (total {total:.2f} s)")
        self.stage_start = time.perf_counter()
        return elapsed

    def total(self):
        return time.perf_counter() - self.image_start


# ============================================================
# FILE HELPERS
# ============================================================

def ensure_odd(value):
    value = max(3, int(round(value)))
    return value + 1 if value % 2 == 0 else value


def read_image(path, flags=cv2.IMREAD_COLOR):
    path = Path(path)
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        if data.size == 0:
            return None
        return cv2.imdecode(data, flags)
    except Exception:
        return None


def write_image(path, image, params=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    params = [] if params is None else params
    success, encoded = cv2.imencode(path.suffix, image, params)
    if not success:
        raise IOError(f"Could not write image: {path}")
    encoded.tofile(str(path))


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(
        [
            p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
        ],
        key=lambda p: p.name.lower(),
    )


def parse_filename_metadata(filename):
    """Extract sample and slice from names such as S1-3rd Wash_section_10.png."""
    match = re.search(
        r"^\s*S\s*(\d+)\s*[-_].*?section[_\s-]*(\d+)",
        Path(filename).stem,
        flags=re.IGNORECASE,
    )
    if match is None:
        raise ValueError(
            "Filename does not match the expected pattern "
            "S<sample>-..._section_<slice>.<ext>: "
            f"{filename}"
        )
    return int(match.group(1)), int(match.group(2))


# ============================================================
# MASK HELPERS
# ============================================================

def normalize_mask(mask):
    if mask is None:
        return None
    if mask.ndim == 3:
        mask = np.max(mask, axis=2)
    return np.where(mask > 0, 255, 0).astype(np.uint8)


def find_matching_mask(directory, image_stem, suffix):
    directory = Path(directory)
    if not directory.exists():
        return None
    expected = f"{image_stem}{suffix}".lower()
    for path in directory.iterdir():
        if (
            path.is_file()
            and path.suffix.lower() in SUPPORTED_EXTENSIONS
            and path.stem.lower() == expected
        ):
            return path
    return None


def load_mask(path, image_shape, default_value=0):
    height, width = image_shape[:2]
    if path is None:
        return np.full((height, width), default_value, dtype=np.uint8)
    mask = read_image(path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        return None
    mask = normalize_mask(mask)
    if mask.shape != (height, width):
        mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
        mask = normalize_mask(mask)
    return mask


def create_regions(image_shape, fov_path, artifact_path, clutter_path, focus_path):
    height, width = image_shape[:2]

    if USE_FOV_MASK:
        if fov_path is None:
            if SKIP_IF_FOV_MISSING:
                return None
            fov = np.full((height, width), 255, dtype=np.uint8)
        else:
            fov = load_mask(fov_path, image_shape, default_value=255)
    else:
        fov = np.full((height, width), 255, dtype=np.uint8)

    if fov is None:
        return None

    artifact = np.zeros((height, width), dtype=np.uint8)
    if USE_ARTIFACT_MASK:
        if artifact_path is not None:
            artifact = load_mask(artifact_path, image_shape, default_value=0)
            if artifact is None:
                return None
        elif not CONTINUE_IF_ARTIFACT_MISSING:
            return None

    clutter = np.zeros((height, width), dtype=np.uint8)
    if USE_CLUTTER_MASK:
        if clutter_path is not None:
            clutter = load_mask(clutter_path, image_shape, default_value=0)
            if clutter is None:
                return None
        elif not CONTINUE_IF_CLUTTER_MISSING:
            return None

    focus = np.zeros((height, width), dtype=np.uint8)
    if USE_FOCUS_MASK:
        if focus_path is not None:
            focus = load_mask(focus_path, image_shape, default_value=0)
            if focus is None:
                return None
        elif not CONTINUE_IF_FOCUS_MISSING:
            return None

    if CLUTTER_MASK_DILATION_RADIUS_PX > 0 and np.any(clutter > 0):
        radius = int(CLUTTER_MASK_DILATION_RADIUS_PX)
        size = 2 * radius + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        clutter = cv2.dilate(clutter, kernel, iterations=1)

    valid = np.where((fov > 0) & (artifact == 0), 255, 0).astype(np.uint8)

    if FOV_INNER_MARGIN_PX > 0:
        size = 2 * FOV_INNER_MARGIN_PX + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        valid = cv2.erode(valid, kernel, iterations=1)

    focus_region = np.where((focus > 0) & (valid > 0), 255, 0).astype(np.uint8)
    clutter_region = np.where(
        (clutter > 0) & (focus_region == 0) & (valid > 0),
        255,
        0,
    ).astype(np.uint8)
    normal_region = np.where(
        (valid > 0) & (focus_region == 0) & (clutter_region == 0),
        255,
        0,
    ).astype(np.uint8)

    return {
        "fov": fov,
        "artifact": artifact,
        "clutter": clutter,
        "focus": focus,
        "valid": valid,
        "normal_region": normal_region,
        "clutter_region": clutter_region,
        "focus_region": focus_region,
    }


# ============================================================
# RESPONSE IMAGE
# ============================================================

def create_gray_and_response(image_bgr, valid_region):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    smoothed = cv2.GaussianBlur(gray, (0, 0), GAUSSIAN_SIGMA)
    background = cv2.medianBlur(smoothed, ensure_odd(BACKGROUND_KERNEL))
    response = background.astype(np.float32) - smoothed.astype(np.float32)
    response = np.maximum(response, 0.0)
    response[valid_region == 0] = 0.0
    return gray, smoothed, background, response


def response_to_display(response, valid_region):
    values = response[(valid_region > 0) & (response > 0)]
    if values.size == 0:
        return np.zeros(response.shape, dtype=np.uint8)
    high = float(np.percentile(values, RESPONSE_DISPLAY_HIGH_PERCENTILE))
    high = max(high, 1.0)
    output = np.clip(response / high, 0.0, 1.0) * 255.0
    output[valid_region == 0] = 0
    return output.astype(np.uint8)


# ============================================================
# RESPONSE-SPOT MEASUREMENTS
# ============================================================

def disk_and_ring_values(image, region_mask, x, y, radius):
    h, w = image.shape
    outer = max(4, int(np.ceil(radius * RING_OUTER_FACTOR)))
    x0, x1 = max(0, x - outer), min(w, x + outer + 1)
    y0, y1 = max(0, y - outer), min(h, y + outer + 1)

    patch = image[y0:y1, x0:x1]
    allowed = region_mask[y0:y1, x0:x1] > 0
    yy, xx = np.ogrid[:patch.shape[0], :patch.shape[1]]
    lx, ly = x - x0, y - y0
    distance = np.sqrt((xx - lx) ** 2 + (yy - ly) ** 2)

    center = distance <= max(1.0, radius * CENTER_RADIUS_FACTOR)
    ring = (
        (distance >= radius * RING_INNER_FACTOR)
        & (distance <= radius * RING_OUTER_FACTOR)
    )

    center_values = patch[center & allowed]
    ring_values = patch[ring & allowed]
    if center_values.size == 0 or ring_values.size == 0:
        return None

    return {
        "center_mean": float(np.mean(center_values)),
        "center_median": float(np.median(center_values)),
        "ring_mean": float(np.mean(ring_values)),
        "ring_median": float(np.median(ring_values)),
    }


def snap_to_response_peak(response, region_mask, x, y, radius=PEAK_SNAP_RADIUS_PX):
    h, w = response.shape
    x0, x1 = max(0, x - radius), min(w, x + radius + 1)
    y0, y1 = max(0, y - radius), min(h, y + radius + 1)
    patch = response[y0:y1, x0:x1].copy()
    allowed = region_mask[y0:y1, x0:x1] > 0
    patch[~allowed] = -1.0
    if patch.size == 0 or np.max(patch) < 0:
        return x, y
    py, px = np.unravel_index(np.argmax(patch), patch.shape)
    return x0 + int(px), y0 + int(py)


def hessian_blob_ratio(dxx, dyy, dxy, x, y):
    # Bright compact spots have two negative curvatures of comparable size.
    a = float(dxx[y, x])
    b = float(dxy[y, x])
    c = float(dyy[y, x])
    trace = a + c
    disc = max(0.0, (a - c) ** 2 + 4.0 * b * b)
    root = np.sqrt(disc)
    eig1 = 0.5 * (trace - root)
    eig2 = 0.5 * (trace + root)

    # At a bright maximum, both should be negative. Reject saddles/ridges.
    if eig1 >= 0 or eig2 >= 0:
        return 0.0, eig1, eig2

    magnitude1 = abs(eig1)
    magnitude2 = abs(eig2)
    ratio = min(magnitude1, magnitude2) / max(magnitude1, magnitude2, 1e-6)
    return float(ratio), eig1, eig2


def local_support_shape(response, region_mask, x, y, radius, peak_value):
    h, w = response.shape
    patch_radius = max(4, int(np.ceil(radius * SUPPORT_PATCH_RADIUS_FACTOR)))
    x0, x1 = max(0, x - patch_radius), min(w, x + patch_radius + 1)
    y0, y1 = max(0, y - patch_radius), min(h, y + patch_radius + 1)

    patch = response[y0:y1, x0:x1]
    allowed = region_mask[y0:y1, x0:x1] > 0
    threshold = max(1.0, float(peak_value) * SUPPORT_THRESHOLD_FRACTION)
    binary = ((patch >= threshold) & allowed).astype(np.uint8)

    lx, ly = x - x0, y - y0
    if not (0 <= ly < binary.shape[0] and 0 <= lx < binary.shape[1]):
        return None
    if binary[ly, lx] == 0:
        return None

    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    label_id = int(labels[ly, lx])
    if label_id <= 0 or label_id >= count:
        return None

    area = int(stats[label_id, cv2.CC_STAT_AREA])
    component = (labels == label_id).astype(np.uint8)
    points = np.column_stack(np.nonzero(component))

    if points.shape[0] < 2:
        axis_ratio = 1.0
    else:
        coords = points[:, [1, 0]].astype(np.float32)
        covariance = np.cov(coords, rowvar=False)
        eigenvalues = np.linalg.eigvalsh(covariance)
        eigenvalues = np.maximum(eigenvalues, 1e-6)
        axis_ratio = float(np.sqrt(eigenvalues[-1] / eigenvalues[0]))

    return {
        "area": area,
        "axis_ratio": axis_ratio,
        "threshold": threshold,
    }


def count_opposite_line_pairs(response, x, y, radius, peak_value):
    h, w = response.shape
    sample_radius = max(2, int(round(radius * DIRECTION_SAMPLE_RADIUS_FACTOR)))
    continuation_threshold = peak_value * DIRECTION_CONTINUATION_FRACTION

    directions = [
        (1, 0),
        (1, 1),
        (0, 1),
        (-1, 1),
    ]

    pair_count = 0
    values = []

    for dx, dy in directions:
        x1 = int(np.clip(x + dx * sample_radius, 0, w - 1))
        y1 = int(np.clip(y + dy * sample_radius, 0, h - 1))
        x2 = int(np.clip(x - dx * sample_radius, 0, w - 1))
        y2 = int(np.clip(y - dy * sample_radius, 0, h - 1))
        v1 = float(response[y1, x1])
        v2 = float(response[y2, x2])
        values.append((v1, v2))
        if v1 >= continuation_threshold and v2 >= continuation_threshold:
            pair_count += 1

    return pair_count, values


# ============================================================
# SPECIAL CLUTTER HELPERS
# ============================================================

def keep_components_by_minimum_area(binary_mask, minimum_area):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        (binary_mask > 0).astype(np.uint8),
        connectivity=8,
    )
    output = np.zeros_like(binary_mask, dtype=np.uint8)
    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area >= minimum_area:
            output[labels == label] = 255
    return output


def measure_clutter_gray_contrast(gray, region_mask, x, y, radius):
    h, w = gray.shape
    outer = max(3, int(np.ceil(radius * 1.50)))
    x0, x1 = max(0, x - outer), min(w, x + outer + 1)
    y0, y1 = max(0, y - outer), min(h, y + outer + 1)
    patch = gray[y0:y1, x0:x1]
    allowed = region_mask[y0:y1, x0:x1] > 0
    yy, xx = np.ogrid[:patch.shape[0], :patch.shape[1]]
    lx, ly = x - x0, y - y0
    distance = np.sqrt((xx - lx) ** 2 + (yy - ly) ** 2)
    inner = distance <= max(1.0, radius * 0.65)
    ring = (distance >= radius * 1.20) & (distance <= outer)
    inner_values = patch[inner & allowed]
    ring_values = patch[ring & allowed]
    if inner_values.size == 0 or ring_values.size == 0:
        return 0.0
    return float(np.median(ring_values) - np.median(inner_values))


def measure_clutter_response_difference(response, region_mask, x, y, radius):
    h, w = response.shape
    outer = max(3, int(np.ceil(radius * 1.45)))
    x0, x1 = max(0, x - outer), min(w, x + outer + 1)
    y0, y1 = max(0, y - outer), min(h, y + outer + 1)
    patch = response[y0:y1, x0:x1]
    allowed = region_mask[y0:y1, x0:x1] > 0
    yy, xx = np.ogrid[:patch.shape[0], :patch.shape[1]]
    lx, ly = x - x0, y - y0
    distance = np.sqrt((xx - lx) ** 2 + (yy - ly) ** 2)
    inner = distance <= max(1.0, radius)
    ring = (distance >= radius * 1.20) & (distance <= outer)
    inner_values = patch[inner & allowed]
    ring_values = patch[ring & allowed]
    if inner_values.size == 0:
        return 0.0, 0.0, 0.0
    inner_mean = float(np.mean(inner_values))
    ring_mean = float(np.mean(ring_values)) if ring_values.size > 0 else 0.0
    return inner_mean, ring_mean, inner_mean - ring_mean


def measure_clutter_peak_prominence(response, x, y):
    h, w = response.shape
    x0, x1 = max(0, x - 1), min(w, x + 2)
    y0, y1 = max(0, y - 1), min(h, y + 2)
    patch = response[y0:y1, x0:x1]
    if patch.size <= 1:
        return 0.0
    ly, lx = y - y0, x - x0
    center = float(patch[ly, lx])
    neighbours = patch.reshape(-1)
    center_index = ly * patch.shape[1] + lx
    neighbours = np.delete(neighbours, center_index)
    return center - float(np.mean(neighbours)) if neighbours.size else 0.0


def create_special_clutter_foreground(clutter_peak_response, clutter_region):
    foreground = np.where(
        (clutter_peak_response >= CLUTTER_FOREGROUND_RESPONSE_THRESHOLD)
        & (clutter_region > 0),
        255,
        0,
    ).astype(np.uint8)

    if CLUTTER_CLOSE_RADIUS_PX > 0:
        radius = int(CLUTTER_CLOSE_RADIUS_PX)
        size = 2 * radius + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        foreground = cv2.morphologyEx(foreground, cv2.MORPH_CLOSE, kernel)

    foreground = keep_components_by_minimum_area(
        foreground,
        CLUTTER_MIN_FOREGROUND_COMPONENT_AREA_PX,
    )
    foreground[clutter_region == 0] = 0
    return foreground


def detect_special_clutter_beads(
    gray,
    response,
    clutter_peak_response,
    clutter_region,
    clutter_foreground,
):
    """Dedicated dense-region detector used only inside the clutter mask."""
    if np.count_nonzero(clutter_region) == 0:
        return [], np.zeros_like(clutter_region, dtype=np.uint8), []

    component_count, component_labels = cv2.connectedComponents(
        (clutter_foreground > 0).astype(np.uint8),
        connectivity=8,
    )

    detections = []
    rejected = []
    marker_image = np.zeros_like(clutter_region, dtype=np.int32)
    next_marker_id = 1

    for component_id in range(1, component_count):
        component = component_labels == component_id
        component_response = np.where(component, clutter_peak_response, 0.0)

        coordinates = peak_local_max(
            component_response,
            min_distance=CLUTTER_PEAK_MIN_DISTANCE_PX,
            threshold_abs=CLUTTER_PEAK_MIN_RESPONSE,
            threshold_rel=CLUTTER_PEAK_THRESHOLD_REL,
            labels=component.astype(np.uint8),
            exclude_border=False,
            num_peaks=CLUTTER_MAX_PEAKS_PER_COMPONENT,
        )

        for y, x in coordinates:
            x, y = int(x), int(y)
            if clutter_region[y, x] == 0:
                continue

            center_response = float(response[y, x])
            record = {
                "x": x,
                "y": y,
                "radius_px": TYPICAL_BEAD_RADIUS_PX,
                "diameter_px": 2.0 * TYPICAL_BEAD_RADIUS_PX,
                "diameter_um": 2.0 * TYPICAL_BEAD_RADIUS_PX * MICRONS_PER_PIXEL,
                "response": center_response,
                "source": "clutter",
            }

            if center_response < CLUTTER_PEAK_MIN_RESPONSE:
                record["rejection_reason"] = "weak_clutter_peak"
                rejected.append(record)
                continue

            if int(gray[y, x]) > CLUTTER_MAX_CENTER_GRAY:
                record["rejection_reason"] = "bright_clutter_center"
                rejected.append(record)
                continue

            prominence = measure_clutter_peak_prominence(
                clutter_peak_response, x, y
            )
            if prominence < CLUTTER_MIN_PEAK_PROMINENCE:
                record["rejection_reason"] = "low_clutter_prominence"
                rejected.append(record)
                continue

            local_contrast = measure_clutter_gray_contrast(
                gray, clutter_region, x, y, TYPICAL_BEAD_RADIUS_PX
            )
            if local_contrast < CLUTTER_MIN_LOCAL_CONTRAST:
                record["rejection_reason"] = "low_clutter_contrast"
                rejected.append(record)
                continue

            _, _, response_difference = measure_clutter_response_difference(
                response, clutter_region, x, y, TYPICAL_BEAD_RADIUS_PX
            )
            if response_difference < CLUTTER_MIN_RESPONSE_DIFFERENCE:
                record["rejection_reason"] = "low_clutter_response_difference"
                rejected.append(record)
                continue

            marker_image[y, x] = next_marker_id
            score = (
                center_response
                + max(prominence, 0.0)
                + max(local_contrast, 0.0)
            )
            record.update({
                "marker_id": next_marker_id,
                "local_contrast": local_contrast,
                "response_difference": response_difference,
                "peak_prominence": prominence,
                "score": score,
                "rejection_reason": "",
            })
            detections.append(record)
            next_marker_id += 1

    if not detections:
        return [], np.zeros_like(clutter_region, dtype=np.uint8), rejected

    watershed_labels = watershed(
        -clutter_peak_response,
        markers=marker_image,
        mask=(clutter_foreground > 0),
    )

    accepted_marker_ids = set()
    for region in regionprops(watershed_labels):
        area = int(region.area)
        if (
            CLUTTER_MIN_WATERSHED_AREA_PX
            <= area
            <= CLUTTER_MAX_WATERSHED_AREA_PX
        ):
            accepted_marker_ids.add(int(region.label))

    filtered = []
    for detection in detections:
        if detection["marker_id"] in accepted_marker_ids:
            filtered.append(detection)
        else:
            rejected_record = dict(detection)
            rejected_record["rejection_reason"] = "clutter_watershed_area"
            rejected.append(rejected_record)

    accepted_mask = np.where(
        np.isin(watershed_labels, list(accepted_marker_ids)),
        255,
        0,
    ).astype(np.uint8)

    return filtered, accepted_mask, rejected


# ============================================================
# RESPONSE-ONLY DETECTOR
# ============================================================

def detect_response_spots(response, response_display, region_mask, source_name):
    settings = REGION_SETTINGS[source_name]
    if np.count_nonzero(region_mask) == 0:
        return [], []

    response_input = (response / 255.0).astype(np.float32)
    response_input *= (region_mask > 0)

    blobs = blob_log(
        response_input,
        min_sigma=DETECT_MIN_SIGMA,
        max_sigma=DETECT_MAX_SIGMA,
        num_sigma=DETECT_NUM_SIGMA,
        threshold=settings["log_threshold"],
        overlap=DETECT_OVERLAP,
        exclude_border=False,
    )

    # Dense clutter supplement: LoG can merge several touching beads into one
    # candidate. Inside clutter masks only, add raw-response local maxima as
    # extra candidate centres. They still pass all validation tests below.
    candidate_rows = [tuple(row) for row in blobs]

    if settings.get("use_peak_supplement", False):
        peak_coordinates = peak_local_max(
            response,
            min_distance=int(settings.get("peak_min_distance", 1)),
            threshold_abs=float(settings.get("peak_threshold_raw", 2.5)),
            labels=(region_mask > 0).astype(np.uint8),
            exclude_border=False,
            num_peaks=int(settings.get("peak_max_candidates", 12000)),
        )

        default_sigma = max(
            DETECT_MIN_SIGMA,
            min(DETECT_MAX_SIGMA, 1.0),
        )

        for peak_y, peak_x in peak_coordinates:
            candidate_rows.append(
                (float(peak_y), float(peak_x), float(default_sigma))
            )

    # Remove near-identical candidate centres before the expensive validation.
    if candidate_rows:
        candidate_rows = sorted(
            candidate_rows,
            key=lambda row: float(response[int(round(row[0])), int(round(row[1]))]),
            reverse=True,
        )

        deduplicated_rows = []
        occupied = set()
        for row in candidate_rows:
            yy = int(round(row[0]))
            xx = int(round(row[1]))
            key = (yy, xx)
            if key in occupied:
                continue
            occupied.add(key)
            deduplicated_rows.append(row)
        candidate_rows = deduplicated_rows

    response_smoothed = cv2.GaussianBlur(response, (0, 0), 0.8)
    dxx = cv2.Sobel(response_smoothed, cv2.CV_32F, 2, 0, ksize=3)
    dyy = cv2.Sobel(response_smoothed, cv2.CV_32F, 0, 2, ksize=3)
    dxy = cv2.Sobel(response_smoothed, cv2.CV_32F, 1, 1, ksize=3)
    h, w = response.shape
    accepted = []
    rejected = []

    for y_float, x_float, sigma in candidate_rows:
        x = int(round(x_float))
        y = int(round(y_float))
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        if region_mask[y, x] == 0:
            continue

        radius = float(np.sqrt(2.0) * sigma)
        diameter = 2.0 * radius
        if diameter < MIN_BEAD_DIAMETER_PX * 0.60:
            continue
        if diameter > MAX_BEAD_DIAMETER_PX * 1.40:
            continue

        x, y = snap_to_response_peak(response, region_mask, x, y)
        peak_value = float(response[y, x])
        display_value = int(response_display[y, x])

        record = {
            "x": x,
            "y": y,
            "radius_px": radius,
            "diameter_px": diameter,
            "diameter_um": diameter * MICRONS_PER_PIXEL,
            "response": peak_value,
            "response_display": display_value,
            "source": source_name,
        }

        if peak_value < settings["min_raw_peak"]:
            record["rejection_reason"] = "weak_raw_peak"
            rejected.append(record)
            continue

        if display_value < settings["min_display_value"]:
            record["rejection_reason"] = "not_visibly_bright"
            rejected.append(record)
            continue

        measurements = disk_and_ring_values(response, region_mask, x, y, radius)
        if measurements is None:
            record["rejection_reason"] = "invalid_center_ring"
            rejected.append(record)
            continue

        center_mean = measurements["center_mean"]
        ring_mean = measurements["ring_mean"]
        center_minus_ring = center_mean - ring_mean
        center_ring_ratio = center_mean / max(ring_mean, 0.25)

        record.update({
            "center_response_mean": center_mean,
            "ring_response_mean": ring_mean,
            "center_minus_ring": center_minus_ring,
            "center_ring_ratio": center_ring_ratio,
        })

        if center_mean < settings["min_center_mean"]:
            record["rejection_reason"] = "weak_center_mean"
            rejected.append(record)
            continue

        # Use both an absolute and relative separation requirement.
        if (
            center_minus_ring < settings["min_center_minus_ring"]
            or center_ring_ratio < settings["min_center_ring_ratio"]
        ):
            record["rejection_reason"] = "not_localized_above_ring"
            rejected.append(record)
            continue

        hessian_ratio, eig1, eig2 = hessian_blob_ratio(dxx, dyy, dxy, x, y)
        record.update({
            "hessian_blob_ratio": hessian_ratio,
            "hessian_eigenvalue_1": eig1,
            "hessian_eigenvalue_2": eig2,
        })

        if hessian_ratio < settings["min_hessian_ratio"]:
            record["rejection_reason"] = "ridge_like_hessian"
            rejected.append(record)
            continue

        support = local_support_shape(response, region_mask, x, y, radius, peak_value)
        if support is None:
            record["rejection_reason"] = "invalid_support"
            rejected.append(record)
            continue

        record.update({
            "support_area_px": support["area"],
            "support_axis_ratio": support["axis_ratio"],
            "support_threshold": support["threshold"],
        })

        if support["area"] < MIN_SUPPORT_AREA_PX:
            record["rejection_reason"] = "support_too_small"
            rejected.append(record)
            continue

        if support["area"] > MAX_SUPPORT_AREA_PX:
            record["rejection_reason"] = "support_too_large"
            rejected.append(record)
            continue

        if support["axis_ratio"] > settings["max_support_axis_ratio"]:
            record["rejection_reason"] = "elongated_support"
            rejected.append(record)
            continue

        opposite_pairs, direction_values = count_opposite_line_pairs(
            response,
            x,
            y,
            radius,
            peak_value,
        )
        record["opposite_line_pairs"] = opposite_pairs

        if opposite_pairs > settings["max_opposite_line_pairs"]:
            record["rejection_reason"] = "continuous_line_response"
            rejected.append(record)
            continue

        # Final score prioritizes genuinely strong, compact response spots.
        record["score"] = (
            peak_value
            + 2.0 * center_minus_ring
            + 10.0 * hessian_ratio
            - 0.25 * support["axis_ratio"]
        )
        record["rejection_reason"] = ""
        accepted.append(record)

    return accepted, rejected


# ============================================================
# DUPLICATE MERGING
# ============================================================

def merge_detections_kdtree(detections, minimum_distance):
    if not detections:
        return []

    detections = sorted(
        detections,
        key=lambda d: (
            SOURCE_PRIORITY.get(d.get("source", ""), 0),
            float(d.get("score", 0.0)),
        ),
        reverse=True,
    )

    coordinates = np.array([[d["x"], d["y"]] for d in detections], dtype=np.float32)
    tree = cKDTree(coordinates)
    removed = np.zeros(len(detections), dtype=bool)
    accepted = []

    for index, detection in enumerate(detections):
        if removed[index]:
            continue
        accepted.append(detection)
        for neighbor in tree.query_ball_point(coordinates[index], r=minimum_distance):
            if neighbor > index:
                removed[neighbor] = True

    accepted = sorted(accepted, key=lambda d: (d["y"], d["x"]))
    for bead_id, detection in enumerate(accepted, start=1):
        detection["id"] = bead_id
    return accepted


# ============================================================
# OUTPUT GENERATION
# ============================================================

def create_bead_mask(image_shape, detections, valid_region):
    h, w = image_shape[:2]
    output = np.zeros((h, w), dtype=np.uint8)
    for detection in detections:
        if USE_ESTIMATED_MASK_RADIUS:
            radius = int(round(detection["radius_px"]))
        else:
            radius = FIXED_MASK_RADIUS
        radius = int(np.clip(radius, MIN_OUTPUT_MASK_RADIUS, MAX_OUTPUT_MASK_RADIUS))
        cv2.circle(output, (int(detection["x"]), int(detection["y"])), radius, 255, -1)
    output[valid_region == 0] = 0
    return output


def add_count_banner(image, count):
    """Add a bottom banner that always fits narrow segmented images."""
    output = image.copy()
    height, width = output.shape[:2]
    text = f"Count: {count}"

    padding_x = max(4, int(round(width * 0.025)))
    padding_y = max(4, int(round(height * 0.004)))
    available_width = max(20, width - 2 * padding_x)

    # Start from the configured scale and reduce it until the text fits.
    font_scale = float(FONT_SCALE)
    thickness = int(FONT_THICKNESS)
    minimum_scale = 0.22

    while font_scale > minimum_scale:
        (text_width, text_height), baseline = cv2.getTextSize(
            text, FONT, font_scale, thickness
        )
        if text_width <= available_width:
            break
        font_scale *= 0.90
        if font_scale < 0.55:
            thickness = 1

    (text_width, text_height), baseline = cv2.getTextSize(
        text, FONT, font_scale, thickness
    )

    banner_height = min(
        height,
        text_height + baseline + 2 * padding_y,
    )
    y0 = max(0, height - banner_height)

    layer = output.copy()
    cv2.rectangle(
        layer,
        (0, y0),
        (width - 1, height - 1),
        (255, 255, 255),
        -1,
    )
    cv2.addWeighted(
        layer,
        BANNER_OPACITY,
        output,
        1.0 - BANNER_OPACITY,
        0,
        output,
    )

    text_x = max(padding_x, (width - text_width) // 2)
    text_y = min(
        height - padding_y - baseline,
        y0 + padding_y + text_height,
    )

    cv2.putText(
        output,
        text,
        (text_x, text_y),
        FONT,
        font_scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA,
    )
    return output


def create_overlay(image_bgr, detections, count):
    overlay = image_bgr.copy()
    for detection in detections:
        cv2.circle(
            overlay,
            (int(detection["x"]), int(detection["y"])),
            CENTER_RADIUS,
            CENTER_COLOR,
            -1,
            cv2.LINE_AA,
        )
    return add_count_banner(overlay, count)


def create_source_overlay(image_bgr, detections):
    colors = {
        "normal": (255, 0, 0),
        "clutter": (0, 255, 0),
        "focus": (255, 0, 255),
    }
    overlay = image_bgr.copy()
    for detection in detections:
        color = colors.get(detection.get("source", ""), (255, 255, 255))
        cv2.circle(
            overlay,
            (int(detection["x"]), int(detection["y"])),
            3,
            color,
            1,
            cv2.LINE_AA,
        )
    return overlay


def create_rejected_overlay(image_bgr, rejected):
    overlay = image_bgr.copy()
    for detection in rejected:
        cv2.circle(
            overlay,
            (int(detection["x"]), int(detection["y"])),
            1,
            (255, 255, 0),
            -1,
            cv2.LINE_AA,
        )
    return overlay


# ============================================================
# DEBUG OUTPUT
# ============================================================

def save_debug_outputs(
    image_path,
    image_bgr,
    regions,
    background,
    response_display,
    strong_response_mask,
    clutter_foreground,
    clutter_watershed_mask,
    normal_detections,
    clutter_detections,
    focus_detections,
    rejected,
    final_detections,
    bead_mask,
    final_overlay,
):
    output_folder = DEBUG_DIR / image_path.stem
    output_folder.mkdir(parents=True, exist_ok=True)

    images = {
        "01_original.jpg": image_bgr,
        "02_fov.png": regions["fov"],
        "03_artifact.png": regions["artifact"],
        "04_clutter.png": regions["clutter_region"],
        "05_focus.png": regions["focus_region"],
        "06_normal_region.png": regions["normal_region"],
        "07_background.png": background,
        "08_response_used_for_detection.png": response_display,
        "09_strong_response_mask.png": strong_response_mask,
        "10_clutter_foreground.png": clutter_foreground,
        "11_clutter_watershed.png": clutter_watershed_mask,
        "12_normal_detections.jpg": create_source_overlay(image_bgr, normal_detections),
        "13_clutter_detections.jpg": create_source_overlay(image_bgr, clutter_detections),
        "14_focus_detections.jpg": create_source_overlay(image_bgr, focus_detections),
        "15_rejected_response_candidates.jpg": create_rejected_overlay(image_bgr, rejected),
        "16_accepted_sources.jpg": create_source_overlay(image_bgr, final_detections),
        "17_bead_mask.png": bead_mask,
        "18_final_overlay.jpg": final_overlay,
    }

    for filename, image in images.items():
        path = output_folder / filename
        if path.suffix.lower() == ".jpg":
            write_image(path, image, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        else:
            write_image(path, image)


def show_debug(image_bgr, response_display, strong_response_mask, final_overlay):
    """
    Matplotlib is intentionally not required. Debug images are saved to disk.
    This avoids Pillow/Matplotlib DLL issues in Windows Anaconda environments.
    """
    if SHOW_DEBUG_IMAGES:
        print(
            "    Debug display skipped: open the saved images in the debug "
            "folder. Matplotlib is not required by this script."
        )


# ============================================================
# PROCESS ONE IMAGE
# ============================================================

def process_image(image_path):
    image_path = Path(image_path)
    timer = StageTimer(image_path.name)

    print("\n" + "=" * 78)
    print(f"Image: {image_path.name}")
    print("=" * 78)

    sample_number, slice_number = parse_filename_metadata(image_path.name)

    result = {
        "Filename": image_path.name,
        "Sample": sample_number,
        "Slice": slice_number,
        "Count": np.nan,
        "Normal count": np.nan,
        "Clutter count": np.nan,
        "Focus count": np.nan,
        "Rejected candidates": np.nan,
        "Processing time (s)": np.nan,
        "Status": "",
    }

    image_bgr = read_image(image_path, cv2.IMREAD_COLOR)
    timer.report("Image loading")
    if image_bgr is None:
        result["Status"] = "image_read_failed"
        return result


    fov_path = find_matching_mask(FOV_MASK_DIR, image_path.stem, FOV_MASK_SUFFIX)
    artifact_path = find_matching_mask(
        ARTIFACT_MASK_DIR, image_path.stem, ARTIFACT_MASK_SUFFIX
    )
    clutter_path = find_matching_mask(
        CLUTTER_MASK_DIR, image_path.stem, CLUTTER_MASK_SUFFIX
    )
    focus_path = find_matching_mask(FOCUS_MASK_DIR, image_path.stem, FOCUS_MASK_SUFFIX)

    print(f"    FOV mask: {fov_path.name if fov_path else 'none'}")
    print(f"    Artifact mask: {artifact_path.name if artifact_path else 'none'}")
    print(f"    Clutter mask: {clutter_path.name if clutter_path else 'none'}")
    print(f"    Focus mask: {focus_path.name if focus_path else 'none'}")

    regions = create_regions(
        image_bgr.shape,
        fov_path,
        artifact_path,
        clutter_path,
        focus_path,
    )
    timer.report("Mask loading and region creation")
    if regions is None:
        result["Status"] = "region_creation_failed"
        return result
    if np.count_nonzero(regions["valid"]) == 0:
        result["Status"] = "empty_valid_region"
        return result

    gray, smoothed, background, response = create_gray_and_response(
        image_bgr, regions["valid"]
    )
    response_display = response_to_display(response, regions["valid"])
    timer.report("Response creation")

    # This mask is only for diagnostics. It shows which visibly strong response
    # pixels are even eligible for the detector's hard brightness checks.
    minimum_display = min(s["min_display_value"] for s in REGION_SETTINGS.values())
    strong_response_mask = np.where(
        (response_display >= minimum_display) & (regions["valid"] > 0),
        255,
        0,
    ).astype(np.uint8)

    normal_detections, normal_rejected = detect_response_spots(
        response,
        response_display,
        regions["normal_region"],
        "normal",
    )
    timer.report("Normal response-spot detector")

    # Dedicated special clutter route. Only the manually marked clutter
    # pixels use this additional foreground + local peaks + watershed method.
    clutter_peak_response = cv2.GaussianBlur(
        response,
        (0, 0),
        CLUTTER_RESPONSE_SIGMA,
    )
    clutter_peak_response[regions["valid"] == 0] = 0.0

    clutter_foreground = create_special_clutter_foreground(
        clutter_peak_response,
        regions["clutter_region"],
    )

    (
        clutter_detections,
        clutter_watershed_mask,
        clutter_rejected,
    ) = detect_special_clutter_beads(
        gray,
        response,
        clutter_peak_response,
        regions["clutter_region"],
        clutter_foreground,
    )
    timer.report("Special clutter detector")

    focus_detections, focus_rejected = detect_response_spots(
        response,
        response_display,
        regions["focus_region"],
        "focus",
    )
    timer.report("Focus response-spot detector")

    all_detections = normal_detections + clutter_detections + focus_detections
    all_rejected = normal_rejected + clutter_rejected + focus_rejected
    final_detections = merge_detections_kdtree(
        all_detections,
        FINAL_DUPLICATE_DISTANCE_PX,
    )
    timer.report("Duplicate merging")

    # One authoritative count is used everywhere: Excel, console, and overlay.
    final_count = len(final_detections)

    bead_mask = create_bead_mask(image_bgr.shape, final_detections, regions["valid"])
    final_overlay = create_overlay(image_bgr, final_detections, final_count)

    category_overlay_dir = OVERLAY_DIR
    category_mask_dir = BEAD_MASK_DIR
    category_table_dir = DETECTION_TABLE_DIR
    for folder in [category_overlay_dir, category_mask_dir, category_table_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    write_image(
        category_overlay_dir / f"{image_path.stem}_overlay.jpg",
        final_overlay,
        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
    )
    write_image(
        category_mask_dir / f"{image_path.stem}_bead_mask.png",
        bead_mask,
    )
    pd.DataFrame(final_detections).to_csv(
        category_table_dir / f"{image_path.stem}_detections.csv",
        index=False,
    )
    pd.DataFrame(all_rejected).to_csv(
        category_table_dir / f"{image_path.stem}_rejected.csv",
        index=False,
    )
    timer.report("Saving standard outputs")

    if DEBUG_MODE:
        if SAVE_DEBUG_IMAGES:
            save_debug_outputs(
                image_path,
                image_bgr,
                regions,
                background,
                response_display,
                strong_response_mask,
                clutter_foreground,
                clutter_watershed_mask,
                normal_detections,
                clutter_detections,
                focus_detections,
                all_rejected,
                final_detections,
                bead_mask,
                final_overlay,
            )
        timer.report("Saving debug outputs")
        show_debug(image_bgr, response_display, strong_response_mask, final_overlay)

    final_normal_count = sum(d["source"] == "normal" for d in final_detections)
    final_clutter_count = sum(d["source"] == "clutter" for d in final_detections)
    final_focus_count = sum(d["source"] == "focus" for d in final_detections)
    total_time = timer.total()

    result.update({
        "Count": final_count,
        "Normal count": final_normal_count,
        "Clutter count": final_clutter_count,
        "Focus count": final_focus_count,
        "Rejected candidates": len(all_rejected),
        "Processing time (s)": round(total_time, 3),
        "Status": "processed",
    })

    print("-" * 78)
    print(f"Final normal count: {final_normal_count}")
    print(f"Final clutter count: {final_clutter_count}")
    print(f"Final focus count: {final_focus_count}")
    print(f"Rejected response candidates: {len(all_rejected)}")
    print(f"FINAL COUNT: {final_count}")
    print(f"TOTAL PROCESSING TIME: {total_time:.2f} seconds")
    return result


# ============================================================
# BUILD IMAGE LIST
# ============================================================

images_to_process = []

if DEBUG_MODE:
    if not DEBUG_IMAGE_PATH.exists():
        raise FileNotFoundError(
            "Debug image does not exist:\n"
            f"{DEBUG_IMAGE_PATH.resolve()}"
        )
    images_to_process.append(DEBUG_IMAGE_PATH)
    print("DEBUG MODE ENABLED")
    print("Only one image will be processed:")
    print(DEBUG_IMAGE_PATH)
else:
    images_to_process = list_images(IMAGES_DIR)
    if not images_to_process:
        raise FileNotFoundError(
            f"No images were found in: {IMAGES_DIR.resolve()}"
        )
    print(f"Total images to process: {len(images_to_process)}")


# ============================================================
# RUN PROCESSING
# ============================================================

batch_start = time.perf_counter()
results = []

for image_index, image_path in enumerate(images_to_process, start=1):
    print(f"\nIMAGE {image_index} OF {len(images_to_process)}")
    try:
        image_result = process_image(image_path)
    except KeyboardInterrupt:
        print("\nProcessing interrupted by user.")
        raise
    except Exception as error:
        print(f"\nERROR processing {image_path.name}:")
        print(error)
        traceback.print_exc()
        try:
            sample_number, slice_number = parse_filename_metadata(image_path.name)
        except Exception:
            sample_number, slice_number = np.nan, np.nan
        image_result = {
            "Filename": image_path.name,
            "Sample": sample_number,
            "Slice": slice_number,
                "Count": np.nan,
            "Normal count": np.nan,
            "Clutter count": np.nan,
            "Focus count": np.nan,
            "Rejected candidates": np.nan,
            "Processing time (s)": np.nan,
            "Status": f"error: {type(error).__name__}",
        }
    results.append(image_result)
    print(f"\nCompleted image {image_index} of {len(images_to_process)}")


# ============================================================
# SAVE FINAL EXCEL
# ============================================================

results_dataframe = pd.DataFrame(results)
if not results_dataframe.empty:
    results_dataframe = (
        results_dataframe
        .sort_values(["Sample", "Slice", "Filename"])
        .reset_index(drop=True)
    )

final_counts_dataframe = results_dataframe[
    ["Filename", "Sample", "Slice", "Count"]
].copy()

with pd.ExcelWriter(FINAL_EXCEL_PATH, engine="openpyxl") as writer:
    final_counts_dataframe.to_excel(writer, sheet_name="Final counts", index=False)
    results_dataframe.to_excel(writer, sheet_name="Processing details", index=False)
    workbook = writer.book
    for sheet_name in ["Final counts", "Processing details"]:
        worksheet = workbook[sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter
            maximum_length = max(
                len(str(cell.value) if cell.value is not None else "")
                for cell in column_cells
            )
            worksheet.column_dimensions[column_letter].width = min(
                maximum_length + 2,
                40,
            )


# ============================================================
# FINAL SUMMARY
# ============================================================

batch_time = time.perf_counter() - batch_start
print("\n" + "=" * 78)
print("PROCESSING COMPLETE")
print("=" * 78)
print(final_counts_dataframe.to_string(index=False))
print(f"\nImages attempted: {len(images_to_process)}")
print(f"Total batch time: {batch_time:.2f} seconds")
if len(images_to_process) > 0:
    print(f"Average time per image: {batch_time / len(images_to_process):.2f} seconds")
print(f"\nExcel file saved to:\n{FINAL_EXCEL_PATH.resolve()}")

final_counts_dataframe

Scale: 3.7037 µm/pixel
Expected bead diameter: 2.70–9.45 pixels
Total images to process: 110

IMAGE 1 OF 110

Image: S1-3rd wash_section_01.png
    Image loading: 0.01 s (total 0.01 s)
    FOV mask: S1-3rd wash_section_01_fov_mask.png
    Artifact mask: none
    Clutter mask: none
    Focus mask: none
    Mask loading and region creation: 0.02 s (total 0.03 s)
    Response creation: 0.03 s (total 0.06 s)
    Normal response-spot detector: 0.44 s (total 0.50 s)
    Special clutter detector: 0.01 s (total 0.51 s)
    Focus response-spot detector: 0.00 s (total 0.51 s)
    Duplicate merging: 0.00 s (total 0.51 s)
    Saving standard outputs: 0.03 s (total 0.54 s)
------------------------------------------------------------------------------
Final normal count: 26
Final clutter count: 0
Final focus count: 0
Rejected response candidates: 3
FINAL COUNT: 26
TOTAL PROCESSING TIME: 0.54 seconds

Completed image 1 of 110

IMAGE 2 OF 110

Image: S1-3rd wash_section_02.png
    Image loading: 0.05 

,Filename,Sample,Slice,Count
0,S1-3rd wash_section_01.png,1,1,26
1,S1-3rd wash_section_02.png,1,2,76
2,S1-3rd wash_section_03.png,1,3,88
3,S1-3rd wash_section_04.png,1,4,104
4,S1-3rd wash_section_05.png,1,5,134
...,...,...,...,...
105,S20-3rd wash_section_06.png,20,6,97
106,S20-3rd wash_section_07.png,20,7,168
107,S20-3rd wash_section_08.png,20,8,114
108,S20-3rd wash_section_09.png,20,9,61
